# BloomBench — Extended Benchmark (evaluation pass)

Replication / evaluation notebook. Calls into the library module [`pysephone.benchmarks.bloombench`](../src/pysephone/benchmarks/bloombench/); does **not** run hyperparameter search (see [`bloombench_extended_hpo.ipynb`](bloombench_extended_hpo.ipynb) for that).

Flow:

1. **HP cache check** — print which (dataset, model) pairs have cached `best_params.json` from the HPO notebook and which will fall back to constructor defaults.
2. **Load datasets** — 18 requested, USA-NPN entries with < `MIN_NPN_SAMPLES` dropped at load time.
3. **Run benchmark** — `run_benchmark(...)` does the (seed × dataset × model) loop with model & evaluation caching.
4. **Results** — MAE table, heatmap, best-per-dataset, Friedman + Nemenyi over the full dataset matrix.

Multi-seed: set `SEEDS = [0, 1, 2, 3, 4]` below. Each new seed re-fits but the HP cache is shared (HPO is keyed by `(dataset, model)`, not by seed).

Climate provider: default is AgERA5. Run [`download_agera5.ipynb`](download_agera5.ipynb) once to populate the HDF5 cache before the first call to `run_benchmark`.

## 1. Setup — import shared config

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pysephone.benchmarks.bloombench import (
    MODELS,
    config as bb_config,
    load_bloombench_datasets,
    load_hp_cache,
    run_benchmark,
    run_comparison,
)

# Seeds to evaluate. Add more for multi-seed replication.
SEEDS = [0]

# Model cache invalidation
FORCE_RETRAIN = False
FORCE_RETRAIN_MODELS: set[str] = set()

print(f'Datasets requested:  {len(bb_config.DATASETS_REQUESTED)}')
print(f'Min dataset samples: {bb_config.MIN_DATASET_SAMPLES}')
print(f'Features:            {bb_config.FEATURE_KEYS}')
print(f'Train fraction:      {bb_config.SPLIT_SIZE}')
print(f'Seeds:               {SEEDS}')
print(f'Torch device:        {bb_config.torch_device()}')
print(f'Run name prefix:     {bb_config.RUN_PREFIX}')

## 2. HP cache status

Shows which (dataset, model) pairs have a `best_params.json` from the HPO notebook. Pairs without a cache will fall back to constructor defaults during the run loop — fine but the table makes it explicit so you can spot accidental gaps.

In [ ]:
TUNED_MODELS = ['RandomForest', 'XGBoost', 'CNN1D', 'LSTM', 'Transformer']
hp_rows = []
for ds_name, _ in bb_config.DATASETS_REQUESTED:
    for model_key in TUNED_MODELS:
        cached = load_hp_cache(ds_name, model_key)
        if cached is None:
            hp_rows.append(dict(dataset=ds_name, model=model_key, params='(default)'))
        else:
            hp_rows.append(dict(
                dataset=ds_name, model=model_key,
                params=', '.join(f'{k}={v}' for k, v in sorted(cached.items())),
            ))
hp_status = pd.DataFrame(hp_rows)
n_default = (hp_status['params'] == '(default)').sum()
n_cached  = len(hp_status) - n_default
print(f'HP cache hits: {n_cached} / {len(hp_status)}  ({n_default} pairs will use defaults)')
hp_status

## 3. Load datasets (year-cutoff split, NPN size filter)

First run will download missing climate data — slow but cached afterwards.

In [ ]:
datasets, load_summary = load_bloombench_datasets(verbose=True)
pd.DataFrame(load_summary)

## 4. Fit / load every model on every dataset (per seed)

In [ ]:
results = run_benchmark(
    seeds=SEEDS,
    datasets_dict=datasets,
    force_retrain=FORCE_RETRAIN,
    force_retrain_models=FORCE_RETRAIN_MODELS,
    verbose=True,
)
results

## 5. Test-MAE summary (mean across seeds)

In [ ]:
# Mean test MAE per (dataset, model) across seeds. With a single seed this is
# just the MAE itself; with multiple seeds it's the across-seed mean.
table_mean = (
    results.groupby(['dataset', 'model'])['mae_test'].mean()
           .unstack('model')
           .reindex(index=list(datasets))
           .reindex(columns=list(MODELS))
           .round(2)
)
table_sd = (
    results.groupby(['dataset', 'model'])['mae_test'].std(ddof=1)
           .unstack('model')
           .reindex(index=list(datasets))
           .reindex(columns=list(MODELS))
           .round(2)
)
if len(SEEDS) > 1:
    display(table_mean)
    print('\nStandard deviation across seeds:')
    display(table_sd)
else:
    table_mean

## 6. Heatmap — test MAE per (dataset, model)

Lighter = better (lower MAE). NaN cells are runs that failed.

In [ ]:
vals = table_mean.values.astype(float)
fig, ax = plt.subplots(figsize=(0.7 * vals.shape[1] + 3.0, 0.35 * vals.shape[0] + 2.0))
im = ax.imshow(vals, aspect='auto', cmap='viridis_r')
ax.set_xticks(range(vals.shape[1]))
ax.set_xticklabels(table_mean.columns, rotation=30, ha='right')
ax.set_yticks(range(vals.shape[0]))
ax.set_yticklabels(table_mean.index)
ax.set_title('Test MAE (days) per (dataset, model)')
vmid = np.nanmean(vals)
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isfinite(v):
            continue
        color = 'white' if v > vmid else 'black'
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8, color=color)
fig.colorbar(im, ax=ax, shrink=0.85, label='MAE (days)')
plt.tight_layout()
plt.show()

## 7. Per-dataset best model and per-model mean rank

In [ ]:
best_per_dataset = pd.DataFrame({
    'dataset': table_mean.index,
    'best_model': table_mean.idxmin(axis=1, skipna=True).values,
    'best_mae':  table_mean.min(axis=1, skipna=True).round(2).values,
})
ranks = table_mean.rank(axis=1, ascending=True)
mean_rank = ranks.mean(axis=0).sort_values().round(2)
mean_rank_df = mean_rank.reset_index()
mean_rank_df.columns = ['model', 'mean_rank']

print('Best model per dataset:')
print(best_per_dataset.to_string(index=False))
print('\nMean rank per model (lower = better):')
print(mean_rank_df.to_string(index=False))

## 8. Statistical comparison (Friedman + Nemenyi)

With N ≥ 15 datasets the Friedman omnibus has reasonable power. Pairwise differences are reported via Nemenyi post-hoc.

In [ ]:
from pysephone.evaluation.model_comparison import (
    plot_score_heatmap, plot_nemenyi_heatmap, plot_rank_distribution,
    autorank_report, plot_critical_difference,
)

report = run_comparison(
    seeds=SEEDS,
    models=list(MODELS),
    metric='mae', split='test',
    alpha=0.05, order='ascending',
    save=True, save_plots=True,
)
print(report.summary())

In [ ]:
fig_scores  = plot_score_heatmap(report.scores, metric=report.metric)
fig_nemenyi = plot_nemenyi_heatmap(report)
fig_ranks   = plot_rank_distribution(report.scores, order=report.order)
try:
    ar = autorank_report(report.scores, alpha=report.alpha, order=report.order)
    fig_cd = plot_critical_difference(ar)
except Exception as exc:
    print(f'Critical-difference diagram skipped: {type(exc).__name__}: {exc}')
plt.show()